<a href="https://colab.research.google.com/github/nujudaly/T5/blob/main/Smart_Traffic_Signs_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlink
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.7/537.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.1/872.1 kB 13.2 MB/s eta 0:00:00


In [ ]:
!wget https://pjreddie.com/media/files/yolov3.weights
!wget https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg
!wget https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names


--2024-09-04 18:03:52--  https://pjreddie.com/media/files/yolov3.weights
Resolving pjreddie.com (pjreddie.com)... 162.0.215.52
Connecting to pjreddie.com (pjreddie.com)|162.0.215.52|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248007048 (237M) [application/octet-stream]
Saving to: ‘yolov3.weights’

yolov3.weights      100%[===================>] 236.52M  10.3MB/s    in 9.8s    

2024-09-04 18:04:02 (24.0 MB/s) - ‘yolov3.weights’ saved [248007048/248007048]

--2024-09-04 18:04:02--  https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8342 (8.1K) [text/plain]
Saving to: ‘yolov3.cfg’

yolov3.cfg          100%[===================>]   8.15K  --.-KB/s    in 0s      

2

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # Clone YOLOv5
%cd yolov5
!pip install -r requirements.txt  # Install dependencies


Cloning into 'yolov5'...
remote: Enumerating objects: 16941, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 16941 (delta 70), reused 95 (delta 49), pack-reused 16805 (from 1)
Receiving objects: 100% (16941/16941), 15.69 MiB | 19.59 MiB/s, done.
Resolving deltas: 100% (11608/11608), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: Pillow 9.4.0
    Uninstalling Pillow-9.4.0:
      Successfully uninstalled Pillow-9.4.0


In [ ]:
import cv2
import torch
from PIL import Image
import subprocess
import numpy as np

# Function to get the stream URL using Streamlink
def get_stream_url(page_url):
    """
    Extracts the stream URL using Streamlink for the given page URL.
    Returns the best stream URL or None if extraction fails.
    """
    command = f"streamlink {page_url} best --stream-url"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Error getting stream URL: {result.stderr}")
        return None
    return result.stdout.strip()

# Function to capture an image from the stream URL
def capture_image(stream_url, output_path):
    """
    Captures an image from the given stream URL and saves it to the output path.
    Returns True if successful, otherwise False.
    """
    cap = cv2.VideoCapture(stream_url)

    if not cap.isOpened():
        print("Error: Could not open video stream.")
        return False

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not read frame from video stream.")
        cap.release()
        return False

    # Save the captured frame as an image
    cv2.imwrite(output_path, frame)
    cap.release()

    print(f"Image captured and saved as {output_path}")
    return True

# Function to initialize the YOLOv5 model
def initialize_yolo_v5(model_name='yolov5s'):
    """
    Loads the YOLOv5 model from the ultralytics library.
    Default model is 'yolov5s' (small). You can choose from 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x'.
    """
    model = torch.hub.load('ultralytics/yolov5', model_name)
    return model

# Function to process an image for vehicle detection using YOLOv5
def detect_vehicles_yolov5(image_path, model):
    """
    Processes the image to detect vehicles using the YOLOv5 model.
    Draws bounding boxes around detected vehicles and returns the count.
    """
    # Load image
    img = Image.open(image_path)

    # Inference with YOLOv5
    results = model(img)

    # Display results
    results.show()  # Show the image with bounding boxes

    # Save the processed image
    processed_image_path = image_path.replace(".jpg", "_processed.jpg")
    results.save(save_dir=".")

    # Count the number of vehicles detected
    vehicle_count = sum(1 for x in results.xyxy[0] if x[5] in [2, 3, 5, 7])  # class IDs for car, truck, bus, motorcycle

    print(f"Detected {vehicle_count} vehicles.")
    return vehicle_count

# Example usage of the functions:

# Step 1: Get the stream URL
page_url = "https://videos-3.earthcam.com/fecnetwork/15559.flv/chunklist_w1182577926.m3u8"  # Replace with your stream page URL
stream_url = get_stream_url(page_url)

if stream_url:
    # Step 2: Capture an image from the stream
    image_path = "captured_frame.jpg"
    if capture_image(stream_url, image_path):
        # Step 3: Initialize YOLOv5 model
        model = initialize_yolo_v5('yolov5s')  # You can also choose 'yolov5m', 'yolov5l', etc.

        # Step 4: Detect vehicles in the captured image
        vehicle_count = detect_vehicles_yolov5(image_path, model)


In [ ]:
import os
import cv2
import subprocess
import time
from ultralytics import YOLO

model = YOLO('yolov8n.pt')


In [ ]:
import cv2
import subprocess
import numpy as np

# Function to get the stream URL using Streamlink
def get_stream_url(page_url):
    """
    Extracts the stream URL using Streamlink for the given page URL.
    Returns the best stream URL or None if extraction fails.
    """
    command = f"streamlink {page_url} best --stream-url"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Error getting stream URL: {result.stderr}")
        return None
    return result.stdout.strip()

# Function to capture an image from the stream URL
def capture_image(stream_url, output_path):
    """
    Captures an image from the given stream URL and saves it to the output path.
    Returns True if successful, otherwise False.
    """
    cap = cv2.VideoCapture(stream_url)

    if not cap.isOpened():
        print("Error: Could not open video stream.")
        return False

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not read frame from video stream.")
        cap.release()
        return False

    # Save the captured frame as an image
    cv2.imwrite(output_path, frame)
    cap.release()

    print(f"Image captured and saved as {output_path}")
    return True

# Function to initialize the YOLO model
def initialize_yolo(model_weights, model_cfg, class_file):
    """
    Loads the YOLO model and COCO class names for detection.
    Returns the YOLO net model and class names.
    """
    net = cv2.dnn.readNet(model_weights, model_cfg)

    # Load the class names from COCO dataset
    with open(class_file, "r") as f:
        classes = f.read().strip().split("\n")

    return net, classes

# Function to process an image for vehicle detection
def detect_vehicles(image_path, net, classes, confidence_threshold=0.5, nms_threshold=0.4):
    """
    Processes the image to detect vehicles using the YOLO model.
    Draws bounding boxes around detected vehicles and returns the count.
    """
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image from {image_path}")
        return 0

    # Get image dimensions
    height, width, _ = image.shape

    # Preprocess the image for YOLO
    blob = cv2.dnn.blobFromImage(image, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)

    # Get output layer names
    output_layers = net.getUnconnectedOutLayersNames()

    # Perform forward pass and get detections
    detections = net.forward(output_layers)

    # Initialize lists for bounding boxes, confidences, and class IDs
    boxes = []
    confidences = []
    class_ids = []

    # Loop through detections
    for detection in detections:
        for obj in detection:
            scores = obj[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            # Detect only vehicles (car, truck, bus, motorcycle)
            if confidence > confidence_threshold and class_id in [2, 3, 5, 7]:  # IDs for vehicles in COCO dataset
                center_x = int(obj[0] * width)
                center_y = int(obj[1] * height)
                w = int(obj[2] * width)
                h = int(obj[3] * height)

                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                # Add to lists
                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Apply Non-Maximum Suppression (NMS)
    indices = cv2.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold)

    # Draw bounding boxes for detections that are kept after NMS
    vehicle_count = 0
    for i in indices.flatten():
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        confidence = confidences[i]

        # Draw rectangle around detected vehicle
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, f"{label} {confidence:.2f}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        vehicle_count += 1

    # Save the processed image with bounding boxes
    processed_image_path = image_path.replace(".jpg", "_processed.jpg")
    cv2.imwrite(processed_image_path, image)

    print(f"Processed image saved as {processed_image_path}")
    print(f"Detected {vehicle_count} vehicles.")

    return vehicle_count

# Example usage of the functions:

# Step 1: Get the stream URL
page_url = "your_stream_page_url_here"  # Replace with your stream page URL
stream_url = get_stream_url(page_url)

if stream_url:
    # Step 2: Capture an image from the stream
    image_path = "/content/captured_frame.jpg"
    if capture_image(stream_url, image_path):
        # Step 3: Initialize the YOLO model (provide correct paths to the model files)
        yolo_net, coco_classes = initialize_yolo("/content/yolov3.weights", "/content/yolov3.cfg", "coco.names")

        # Step 4: Detect vehicles in the captured image
        vehicle_count = detect_vehicles(image_path, yolo_net, coco_classes)


In [ ]:
# Function to get the stream URL using Streamlink
def get_stream_url(page_url):
    """
    Extracts the stream URL using Streamlink for the given page URL.
    Returns the best stream URL or None if extraction fails.
    """
    command = f"streamlink {page_url} best --stream-url"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Error getting stream URL: {result.stderr}")
        return None
    return result.stdout.strip()

# Function to capture an image from the stream URL
def capture_image(stream_url, output_path):
    """
    Captures an image from the given stream URL and saves it to the output path.
    Returns True if successful, otherwise False.
    """
    cap = cv2.VideoCapture(stream_url)

    if not cap.isOpened():
        print("Error: Could not open video stream.")
        return False

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not read frame from video stream.")
        cap.release()
        return False

    # Save the captured frame as an image
    cv2.imwrite(output_path, frame)
    cap.release()

    print(f"Image captured and saved as {output_path}")
    return True

# Function to process frames and count vehicles using YOLO
def process_frame(image_path):
    """
    Processes the captured image with YOLOv8 to detect vehicles.
    Draws bounding boxes around detected vehicles and saves the processed image.
    Returns the count of detected vehicles.
    """
    frame = cv2.imread(image_path)
    if frame is None:
        print(f"Error: Could not load image from {image_path}")
        return 0

    results = model(frame)
    vehicle_count = 0

    # Loop through results and detect vehicles
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            conf = box.conf[0]
            cls = int(box.cls[0])
            label = model.names[cls]

            if label in ['car', 'truck', 'bus', 'motorcycle']:
                vehicle_count += 1
                # Draw bounding boxes and labels on the frame
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
                cv2.putText(frame, f'{label} {conf:.2f}', (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    # Save the processed image
    processed_image_path = image_path.replace(".jpg", "_processed.jpg")
    cv2.imwrite(processed_image_path, frame)

    print(f"Processed image saved as {processed_image_path}")
    return vehicle_count


In [ ]:
def adjust_green_signal_time(vehicle_count):
    """
    Adjusts the green signal time based on the detected vehicle count.
    """
    base_green_time = 20
    vehicle_multiplier = 2

    green_time = base_green_time + (vehicle_count * vehicle_multiplier)
    return green_time


In [ ]:
def update_vehicle_count_file(vehicle_count):
    """
    Updates the vehicle count file to store the latest vehicle count.
    """
    with open("vehicle_count.txt", "w") as file:
        file.write(str(vehicle_count))
    print("Vehicle count updated in 'vehicle_count.txt'.")

def read_vehicle_count():
    """
    Reads the vehicle count from the 'vehicle_count.txt' file.
    Returns the vehicle count as an integer or 0 if the file doesn't exist.
    """
    try:
        with open("vehicle_count.txt", "r") as file:
            vehicle_count = int(file.read())
            if vehicle_count < 0:
                raise ValueError("Vehicle count cannot be negative.")
            return vehicle_count
    except (ValueError, FileNotFoundError) as e:
        print(f"Error reading vehicle count from file: {e}")
        return 0


In [ ]:
def process_traffic_stream():
    """
    Captures images from a traffic stream, processes them with YOLO,
    and adjusts the green signal time dynamically based on vehicle detection.
    """
    page_url = "https://videos-3.earthcam.com/fecnetwork/15559.flv/chunklist_w372777735.m3u8"
    stream_url = get_stream_url(page_url)

    if not stream_url:
        print("Error: Could not extract stream URL. Exiting.")
        return

    interval_seconds = 20  # Time interval between captures

    while True:
        timestamp = time.strftime("%Y%m%d%H%M%S")
        image_filename = f"captured_image_{timestamp}.jpg"

        # Capture image from stream
        if not capture_image(stream_url, image_filename):
            print("Stream could not be opened or frame not captured. Exiting.")
            break

        # Process the image with YOLO and get the vehicle count
        vehicle_count = detect_vehicles_yolov5(image_path, model)
        update_vehicle_count_file(vehicle_count)

        print(f"Total vehicles detected: {vehicle_count}")

        # Immediately adjust the green signal based on the current vehicle count
        green_time = adjust_green_signal_time(vehicle_count)
        print(f"Adjusted Green Signal Time: {green_time} seconds")

        # Save the adjusted green signal time to a file
        with open("adjusted_green_time.txt", "w") as output_file:
            output_file.write(f"Green signal adjusted to: {green_time} seconds")

        print(f"Adjusted green time saved in 'adjusted_green_time.txt'.")

        time.sleep(interval_seconds)


In [ ]:
if __name__ == "__main__":
    process_traffic_stream()  # Run the traffic stream processing
    adjust_green_signal_time(vehicle_count)   # Adjust the green signal time based on vehicle count


In [ ]:
import torch
from sklearn.metrics import precision_score, recall_score, average_precision_score
import numpy as np
from tqdm import tqdm

# Function to initialize the YOLOv5 model
def initialize_yolo_v5(model_name='yolov5s'):
    """
    Loads the YOLOv5 model from the ultralytics library.
    Default model is 'yolov5s' (small). You can choose from 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x'.
    """
    model = torch.hub.load('ultralytics/yolov5', model_name)
    return model

# Function to calculate Intersection over Union (IoU)
def calculate_iou(box1, box2):
    x_min1, y_min1, x_max1, y_max1 = box1
    x_min2, y_min2, x_max2, y_max2 = box2

    # Calculate the (x, y)-coordinates of the intersection rectangle
    x_left = max(x_min1, x_min2)
    y_top = max(y_min1, y_min2)
    x_right = min(x_max1, x_max2)
    y_bottom = min(y_min1, y_max2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0  # No overlap

    # Compute the area of intersection rectangle
    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    # Compute the area of both the prediction and ground-truth rectangles
    box1_area = (x_max1 - x_min1) * (y_max1 - y_min1)
    box2_area = (x_max2 - x_min2) * (y_max2 - y_min2)

    # Compute the Intersection over Union (IoU)
    iou = intersection_area / float(box1_area + box2_area - intersection_area)
    return iou

# Function to evaluate YOLO predictions
def evaluate_predictions(predictions, ground_truths, iou_threshold=0.5):
    true_positives = []
    false_positives = []
    false_negatives = []
    all_precisions = []
    all_recalls = []

    for pred, gt in zip(predictions, ground_truths):
        true_pos = 0
        false_pos = 0
        false_neg = len(gt)  # Initially, assume all ground truth objects are not detected

        # Compare predicted bounding boxes with ground truth
        for p_class_id, confidence, p_bbox in pred:
            best_iou = 0
            best_gt = None

            # Compare each predicted box with ground truth boxes
            for gt_class_id, gt_bbox in gt:
                if p_class_id == gt_class_id:  # Check if class matches
                    iou = calculate_iou(p_bbox, gt_bbox)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt = gt_bbox

            # If the IoU is above the threshold, it's a true positive
            if best_iou >= iou_threshold:
                true_pos += 1
                false_neg -= 1  # Correct match, so reduce false negatives
            else:
                false_pos += 1  # False positive (predicted but no match)

        # Calculate Precision and Recall for each image
        precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
        recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0

        all_precisions.append(precision)
        all_recalls.append(recall)
        true_positives.append(true_pos)
        false_positives.append(false_pos)
        false_negatives.append(false_neg)

    mean_precision = np.mean(all_precisions)
    mean_recall = np.mean(all_recalls)

    return mean_precision, mean_recall, np.mean(true_positives)

# Example usage with YOLOv5
model_names = ['yolov5s', 'yolov5m', 'yolov5l']  # Different YOLO models to evaluate

for model_name in model_names:
    print(f"Evaluating model: {model_name}")

    # Initialize YOLO model
    model = initialize_yolo_v5(model_name)

    # Assuming predictions and ground truths are available
    predictions = [
        # Format: [(class_id, confidence, [x_min, y_min, x_max, y_max])]
    ]
    ground_truths = [
        # Format: [(class_id, [x_min, y_min, x_max, y_max])]
    ]

    # Evaluate performance
    mean_precision, mean_recall, mean_iou = evaluate_predictions(predictions, ground_truths)

    print(f"Mean Precision: {mean_precision:.2f}")
    print(f"Mean Recall: {mean_recall:.2f}")
    print(f"Mean IoU: {mean_iou:.2f}")
